# CoNIC Story Visualizer

Use this notebook to inspect matched CoNIC/Lizard patches and generate report-ready figures that combine the original patch, the ground-truth mask, the outline overlay, and optional model-prediction overlays when prediction roots are available locally.


## Configuration

Set the repo-relative inputs here. Keep `PREDICTION_ROOTS` empty for the current GT-only workflow, or point it at local model output directories if predicted masks are available.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

os.environ.setdefault('MPLCONFIGDIR', str((Path.cwd() / 'tmp' / 'matplotlib').resolve()))

import pandas as pd
import matplotlib.pyplot as plt


def find_repo_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current working directory.')


REPO_ROOT = find_repo_root()
VIS_DIR = REPO_ROOT / 'scripts' / 'benchmarking' / 'visualizations'
if str(VIS_DIR) not in sys.path:
    sys.path.insert(0, str(VIS_DIR))

from conic_visualization_utils import (
    PredictionLocator,
    ViewSpec,
    build_matched_patch_table,
    compute_embedding_projection,
    load_embedding_table,
    plot_archetype_gallery,
    plot_dominant_class_atlas,
    plot_metric_caseboards,
    plot_patch_view_grid,
    plot_umap_patch_neighborhoods,
    plot_umap_scatter_grid,
    select_dominant_class_exemplars,
    select_umap_fraction_connective_cases,
    select_umap_story_cases,
)
from build_conic_story_figures import attach_umap_report_columns, build_dominant_class_atlas_map, select_major_dominant_classes, select_story_cases

EVAL_DIR = REPO_ROOT / 'outputs' / 'conic_liz'
METADATA_CSV = REPO_ROOT / 'outputs' / 'conic_liz' / 'embed_morph.csv'
EMBEDDING_CSV = REPO_ROOT / 'outputs' / 'conic_liz' / 'embedding_tables' / 'embed_morph_with_vectors.csv'
DATASET_ROOT = REPO_ROOT / 'data' / 'conic_lizard'
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'conic_liz' / 'analysis' / 'visuals'

PREDICTION_ROOTS = {
    # 'cellpose_sam': REPO_ROOT / 'inference' / 'benchmarking' / 'conic_liz' / 'cellpose_sam',
    # 'cellsam': REPO_ROOT / 'inference' / 'benchmarking' / 'conic_liz' / 'cellsam',
    # 'cellvit_sam': REPO_ROOT / 'inference' / 'benchmarking' / 'conic_liz' / 'cellvit_sam',
    # 'stardist': REPO_ROOT / 'inference' / 'benchmarking' / 'conic_liz' / 'stardist',
}


## Load The Matched Evaluation Table

This table is the same matched-patch view used in the report logic: only patches present for all four models survive the merge.


In [ ]:
matched = build_matched_patch_table(EVAL_DIR, metadata_csv=METADATA_CSV)
case_groups = select_story_cases(matched)
embedding_table = load_embedding_table(EMBEDDING_CSV)
projection_frame = compute_embedding_projection(embedding_table)
projection_frame = attach_umap_report_columns(projection_frame, matched)
dominant_classes = select_major_dominant_classes(projection_frame)
atlas_map = build_dominant_class_atlas_map(projection_frame, dominant_classes=dominant_classes)
umap_cases = select_umap_story_cases(projection_frame)
connective_umap_cases = select_umap_fraction_connective_cases(projection_frame)
prediction_locator = PredictionLocator(prediction_roots=PREDICTION_ROOTS) if PREDICTION_ROOTS else None

print(f'matched rows: {len(matched)}')
print(f'embedding rows: {len(projection_frame)}')
for group_name, cases in case_groups.items():
    print(group_name, [case.sample_id for case in cases])
print('dominant classes', dominant_classes)
print('umap cases', [case.sample_id for case in umap_cases])
print('connective umap cases', [case.sample_id for case in connective_umap_cases])

preview_columns = [
    'sample_id',
    'story_label',
    'pq_median',
    'foreground_fraction',
    'total_nuclei',
    'cellpose_sam__instance_pq',
    'cellsam__instance_pq',
    'cellvit_sam__instance_pq',
    'stardist__instance_pq',
]
display(matched[preview_columns].sort_values('pq_median').head(12))


## Archetype Gallery

This figure works well near the start of the report because it grounds the summary statistics in actual tissue phenotypes.


In [ ]:
fig, _ = plot_archetype_gallery(
    case_groups['archetypes'],
    matched,
    dataset_root=DATASET_ROOT,
    suptitle='Patch archetypes behind the summary statistics',
)
plt.show()


## Dominant-Class Atlas

This atlas works as a visual glossary for the composition section by showing representative patches for the major dominant-class regimes with enough support to matter in the dataset.


In [ ]:
fig, _ = plot_dominant_class_atlas(
    atlas_map,
    dataset_root=DATASET_ROOT,
    suptitle='Major dominant classes as actual patches',
)
plt.show()


## Morphology Contrast Grid

This view is useful in the morphology-linked difficulty section because it shows the original patch, the mask alone, and the outline overlay for a sparse connective patch versus a dense mixed patch.


In [ ]:
fig, _ = plot_patch_view_grid(
    [case.sample_id for case in case_groups['morphology']],
    [
        ViewSpec(kind='image', title='Original patch'),
        ViewSpec(kind='gt_mask', title='Ground-truth mask'),
        ViewSpec(kind='gt_overlay', title='Ground-truth outline'),
    ],
    dataset_root=DATASET_ROOT,
    row_titles=[case.title for case in case_groups['morphology']],
    suptitle='Morphology contrast: sparse connective versus dense mixed tissue',
)
plt.show()


## UMAP Signal Grid

This is the report-ready UMAP grid. It keeps one coordinate system and recolors the same manifold by morphology and consensus patch quality.


In [ ]:
fig, _ = plot_umap_scatter_grid(
    projection_frame,
    [
        ('fraction_connective_level', 'Connective fraction'),
        ('foreground_fraction_level', 'Foreground fraction'),
        ('total_nuclei_level', 'Total nuclei'),
        ('consensus_pq_tier', 'Consensus PQ'),
    ],
    ncols=2,
)
plt.show()


## UMAP Neighborhood Patches

This view is the clearest bridge from embedding space back to actual tissue. It uses signal-aware representative patches rather than a naive centroid pick so the selected crops remain visually meaningful.


In [ ]:
fig, _ = plot_umap_patch_neighborhoods(
    projection_frame,
    umap_cases,
    dataset_root=DATASET_ROOT,
    suptitle=None,
    show_scatter_title=False,
)
plt.show()


## UMAP Connective-Fraction Neighborhoods

This companion view uses the same patch-linked layout but colors the manifold by `fraction_connective_level` using the same low / mid / high / Missing quantile bins as the earlier embedding notebook.


In [ ]:
fig, _ = plot_umap_patch_neighborhoods(
    projection_frame,
    connective_umap_cases,
    dataset_root=DATASET_ROOT,
    color_column='fraction_connective_level',
    suptitle=None,
    show_scatter_title=False,
)
plt.show()


## Caseboards

By default this uses GT-only views plus per-model PQ bars. If you populate `PREDICTION_ROOTS`, the same caseboard function can append predicted-outline columns for the selected models.


In [ ]:
prediction_models = sorted(PREDICTION_ROOTS) if PREDICTION_ROOTS else None
fig, _ = plot_metric_caseboards(
    case_groups['caseboards'],
    matched,
    dataset_root=DATASET_ROOT,
    prediction_locator=prediction_locator,
    prediction_models=prediction_models,
    suptitle='Representative patch caseboards with per-model instance PQ',
)
plt.show()


## Optional Save Section

Set `SAVE_FIGURES = True` to export the current notebook figures to the same analysis folder used by the CLI script.


In [ ]:
SAVE_FIGURES = False

if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    fig1, _ = plot_archetype_gallery(
        case_groups['archetypes'],
        matched,
        dataset_root=DATASET_ROOT,
        suptitle='Patch archetypes behind the summary statistics',
    )
    fig1.savefig(OUTPUT_DIR / 'patch_archetype_gallery_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig1)

    fig2, _ = plot_dominant_class_atlas(
        atlas_map,
        dataset_root=DATASET_ROOT,
        suptitle='Major dominant classes as actual patches',
    )
    fig2.savefig(OUTPUT_DIR / 'dominant_class_patch_atlas_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig2)

    fig3, _ = plot_patch_view_grid(
        [case.sample_id for case in case_groups['morphology']],
        [
            ViewSpec(kind='image', title='Original patch'),
            ViewSpec(kind='gt_mask', title='Ground-truth mask'),
            ViewSpec(kind='gt_overlay', title='Ground-truth outline'),
        ],
        dataset_root=DATASET_ROOT,
        row_titles=[case.title for case in case_groups['morphology']],
        suptitle='Morphology contrast: sparse connective versus dense mixed tissue',
    )
    fig3.savefig(OUTPUT_DIR / 'morphology_contrast_grid_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig3)

    fig4, _ = plot_umap_scatter_grid(
        projection_frame,
        [
            ('fraction_connective_level', 'Connective fraction'),
            ('foreground_fraction_level', 'Foreground fraction'),
            ('total_nuclei_level', 'Total nuclei'),
            ('consensus_pq_tier', 'Consensus PQ'),
        ],
        ncols=2,
    )
    fig4.savefig(OUTPUT_DIR / 'umap_embedding_signal_grid_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig4)

    fig5, _ = plot_umap_patch_neighborhoods(
        projection_frame,
        umap_cases,
        dataset_root=DATASET_ROOT,
        suptitle=None,
        show_scatter_title=False,
    )
    fig5.savefig(OUTPUT_DIR / 'umap_patch_neighborhoods_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig5)

    fig6, _ = plot_umap_patch_neighborhoods(
        projection_frame,
        connective_umap_cases,
        dataset_root=DATASET_ROOT,
        color_column='fraction_connective_level',
        suptitle=None,
        show_scatter_title=False,
    )
    fig6.savefig(OUTPUT_DIR / 'umap_fraction_connective_neighborhoods_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig6)

    fig7, _ = plot_metric_caseboards(
        case_groups['caseboards'],
        matched,
        dataset_root=DATASET_ROOT,
        prediction_locator=prediction_locator,
        prediction_models=prediction_models,
        suptitle='Representative patch caseboards with per-model instance PQ',
    )
    fig7.savefig(OUTPUT_DIR / 'model_caseboards_notebook.png', dpi=200, bbox_inches='tight')
    plt.close(fig7)

    print(f'saved figures to {OUTPUT_DIR}')
else:
    print('SAVE_FIGURES is False; no files were written.')
